In [1]:
# import time
# import torch
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer

# MODEL_NAME = "Qwen/Qwen3-1.7B"
# THINK_END_ID = 151668  # token </think> của Qwen3

# SYSTEM_PROMPT = (
#     "Bạn là trợ lý tóm tắt tin tức tiếng Việt. Hãy viết bản tóm tắt ngắn gọn "
#     "(2-3 câu), giữ lại các thông tin quan trọng và không thêm chi tiết "
#     "không có trong bài báo. Chỉ trả về bản tóm tắt."
# )

# # Cấu hình sinh CỐ ĐỊNH — dùng y hệt cho baseline và sau fine-tune
# GEN_KWARGS = dict(
#     max_new_tokens=256,      # chỉnh lại theo phân vị 95% độ dài abstract
#     do_sample=False,         # greedy, kết quả lặp lại được
#     temperature=None,        # ghi đè sampling mặc định trong generation_config
#     top_p=None,
#     top_k=None,
#     repetition_penalty=1.05,
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype="auto", device_map="auto"
# )
# model.eval()


# def build_prompt(article: str) -> str:
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": f"Nội dung bài báo:\n{article}"},
#     ]
#     return tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=False,  # tắt thinking cho bài toán tóm tắt
#     )


# @torch.no_grad()
# def summarize(article: str) -> str:
#     text = build_prompt(article)
#     inputs = tokenizer([text], return_tensors="pt").to(model.device)
#     out = model.generate(**inputs, **GEN_KWARGS)
#     ids = out[0][inputs.input_ids.shape[1]:].tolist()
#     # phòng trường hợp mô hình vẫn sinh khối <think>...</think>
#     if THINK_END_ID in ids:
#         ids = ids[len(ids) - ids[::-1].index(THINK_END_ID):]
#     return tokenizer.decode(ids, skip_special_tokens=True).strip()


# if __name__ == "__main__":
#     ds = load_dataset("Yuhthe/vietnews")
#     print(ds)  # xác nhận tên split và tên trường trước khi dùng

#     # In prompt đã định dạng 1 lần để kiểm tra chat template
#     sample = ds["test"][0]
#     print("=== PROMPT ===")
#     print(build_prompt(sample["article"][:300] + "..."))

#     for i in range(3):
#         ex = ds["test"][i]
#         n_tok = len(tokenizer(ex["article"]).input_ids)
#         t0 = time.time()
#         pred = summarize(ex["article"])
#         print(f"\n=== Mẫu {i} | article: {n_tok} tokens | {time.time() - t0:.1f}s ===")
#         print("MODEL :", pred)
#         print("THAM CHIẾU:", ex["abstract"])

#     if torch.cuda.is_available():
#         print(f"\nGPU peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [2]:
"""Bước 2: khảo sát dữ liệu VietNews bằng tokenizer của Qwen3."""
import re
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B"
N_SAMPLE = 3000   # lấy mẫu ngẫu nhiên cho nhanh; tăng lên nếu muốn
SEED = 42

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
ds = load_dataset("Yuhthe/vietnews")


NUM_RE = re.compile(r"\d+(?:[.,/:]\d+)*")


def numbers(text):
    """Các con số / ngày tháng, ví dụ 25/2, 3,2, 1994."""
    return set(NUM_RE.findall(text))


def names(text):
    """Cụm >= 2 âm tiết viết hoa liên tiếp (gần đúng tên riêng)."""
    out, cur = set(), []
    for t in text.split():
        if t[0].isupper() and t.replace(".", "").isalpha():
            cur.append(t)
        else:
            if len(cur) >= 2:
                out.add(" ".join(cur))
            cur = []
    if len(cur) >= 2:
        out.add(" ".join(cur))
    return out


def describe(name, arr):
    arr = np.array(arr)
    p = np.percentile(arr, [50, 90, 95, 99])
    print(f"{name:<22} mean={arr.mean():7.1f}  median={p[0]:6.0f}  "
          f"p90={p[1]:6.0f}  p95={p[2]:6.0f}  p99={p[3]:6.0f}  max={arr.max():6.0f}")


for split in ["train", "validation", "test"]:
    d = ds[split]
    # 1. Mẫu rỗng / quá ngắn
    empty = sum(1 for a, b in zip(d["article"], d["abstract"])
                if not a or not b or len(a.strip()) < 50 or len(b.strip()) < 10)
    # 2. Trùng lặp article
    dup = len(d) - len(set(d["article"]))
    print(f"\n===== {split}: {len(d)} mẫu | rỗng/quá ngắn: {empty} | article trùng: {dup}")

    sub = d.shuffle(seed=SEED).select(range(min(N_SAMPLE, len(d))))
    # datasets >= 4 trả về đối tượng Column, không phải list -> ép sang list
    arts = list(sub["article"])
    abss = list(sub["abstract"])
    art_tok = [len(x) for x in tok(arts).input_ids]
    abs_tok = [len(x) for x in tok(abss).input_ids]
    describe("article (tokens)", art_tok)
    describe("abstract (tokens)", abs_tok)
    describe("abstract (số câu)", [max(1, s.count(" . ")) for s in abss])
    describe("tỷ lệ nén art/abs", np.array(art_tok) / np.maximum(1, abs_tok))

    # 3. Rò rỉ: abstract có nằm nguyên trong article không?
    leak = sum(1 for a, b in zip(arts, abss) if b.strip()[:80] in a)
    print(f"abstract (80 ký tự đầu) xuất hiện nguyên văn trong article: {leak}/{len(sub)}")

    # 3b. Abstract có thông tin KHÔNG nằm trong article (số, tên riêng)?
    n_num = n_name = n_any = 0
    examples = []
    for a, b in zip(arts, abss):
        miss_num = [x for x in numbers(b) if x not in a]
        miss_name = [x for x in names(b) if x not in a]
        n_num += bool(miss_num)
        n_name += bool(miss_name)
        n_any += bool(miss_num or miss_name)
        if (miss_num or miss_name) and len(examples) < 5:
            examples.append((miss_num, miss_name, b[:150]))
    n = len(sub)
    print(f"abstract có SỐ không có trong article     : {n_num / n * 100:5.1f}%")
    print(f"abstract có TÊN RIÊNG không có trong article: {n_name / n * 100:5.1f}%")
    print(f"abstract có ít nhất 1 trong 2               : {n_any / n * 100:5.1f}%")
    if split == "train":
        for mn, mname, txt in examples:
            print(f"   số thiếu={mn} | tên thiếu={mname}\n   -> {txt}...")

    # 4. Tỷ lệ bài vượt các ngưỡng max_length thường dùng
    for L in [512, 1024, 1536, 2048]:
        print(f"  article > {L:4d} tokens: {np.mean(np.array(art_tok) > L) * 100:5.1f}%")

# 5. Xem nhanh chủ đề qua tiêu đề ngẫu nhiên
print("\n===== 20 tiêu đề ngẫu nhiên (train)")
for t in ds["train"].shuffle(seed=SEED).select(range(20))["title"]:
    print(" -", t)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (7 outputs left out)

Generating train split:   0%|          | 0/99134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]


===== train: 99134 mẫu | rỗng/quá ngắn: 0 | article trùng: 0
article (tokens)       mean=  655.3  median=   563  p90=  1143  p95=  1319  p99=  1603  max=  2247
abstract (tokens)      mean=   45.8  median=    43  p90=    65  p95=    72  p99=    88  max=   151
abstract (số câu)      mean=    1.0  median=     1  p90=     1  p95=     1  p99=     2  max=     3
tỷ lệ nén art/abs      mean=   14.9  median=    13  p90=    25  p95=    30  p99=    40  max=    60
abstract (80 ký tự đầu) xuất hiện nguyên văn trong article: 62/3000
abstract có SỐ không có trong article     :  13.0%
abstract có TÊN RIÊNG không có trong article:  17.0%
abstract có ít nhất 1 trong 2               :  26.0%
   số thiếu=['2020', '2017'] | tên thiếu=[]
   -> Đề án xây dựng TP. HCM trở thành đô thị thông minh giai đoạn 2017-2020 , tầm nhìn 2030 đang dần hoàn thiện . Người dân TP có thể chờ đợi gì ở đề...
   số thiếu=[] | tên thiếu=['Ông Đỗ Tuấn Khiêm']
   -> Ông Đỗ Tuấn Khiêm bị kỷ luật do thiếu trách nhiệm trong chỉ đạo 

In [3]:
%%writefile /kaggle/working/eval_baseline.py
"""Bước 3-4: ROUGE tiếng Việt + baseline (Lead-1, Qwen3 zero-shot) trên tập test con cố định.

Kết quả lưu trong thư mục results/:
  test_subset_guids.csv   - danh sách guid của tập test con (dùng lại cho E1, E2)
  reference.csv / lead1.csv / zeroshot.csv - đầu ra và điểm từng mẫu
  summary.json            - bảng tổng hợp + thông tin môi trường
"""
import json
import os
import re
import time
import unicodedata

import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

# ========================= CẤU HÌNH =========================
MODEL_NAME = "Qwen/Qwen3-1.7B"
N_TEST = 500
SEED = 42
BATCH_SIZE = 8              # giữ CỐ ĐỊNH giữa các thí nghiệm
MAX_ARTICLE_TOKENS = 1536   # chỉ ~1.5% bài bị cắt
OUT_DIR = "results"
RUN_SYSTEMS = ["lead1", "zeroshot"]

SYSTEM_PROMPT = (
    "Bạn là trợ lý tóm tắt tin tức tiếng Việt. Hãy tóm tắt bài báo bằng đúng 1 câu "
    "(khoảng 30-50 từ), nêu sự kiện chính và chỉ dùng thông tin có trong bài báo. "
    "Chỉ trả về câu tóm tắt."
)
GEN_KWARGS = dict(
    max_new_tokens=128,      # p99 của abstract ~90 token
    do_sample=False,
    temperature=None, top_p=None, top_k=None,
    repetition_penalty=1.05,
)
THINK_END_ID = 151668       # token </think> của Qwen3


# ===================== CHUẨN HOÁ VĂN BẢN =====================
def nfc(text):
    """Đưa dấu tiếng Việt về dạng dựng sẵn (NFC) để so sánh nhất quán."""
    return unicodedata.normalize("NFC", text)


def detok(text):
    """Bỏ khoảng trắng thừa quanh dấu câu của VietNews.
    'Ngày 27/3 , Công an ( SN 1989 ) .' -> 'Ngày 27/3, Công an (SN 1989).'
    Dùng LẠI hàm này khi chuẩn bị dữ liệu fine-tune để input nhất quán."""
    t = nfc(text)
    t = re.sub(r"\s+([,.;:!?)\]])", r"\1", t)
    t = re.sub(r"([(\[])\s+", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()


def words(text):
    """Tách âm tiết (bỏ dấu câu, chữ thường). Đây là đơn vị tính ROUGE."""
    return re.findall(r"\w+", nfc(text).lower())


class ViSyllableTokenizer:
    def tokenize(self, text):
        return words(text)


SCORER = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=False, tokenizer=ViSyllableTokenizer()
)


def rouge(pred, ref):
    s = SCORER.score(ref, pred)   # thứ tự: (tham chiếu, dự đoán)
    return {k: v.fmeasure for k, v in s.items()}


def self_test():
    a = "Ngày 25/2 , TAND TP. Đà Nẵng tuyên phạt Hồ Xuân Huy 12 năm tù ."
    b = "Ngày 25/2, TAND TP. Đà Nẵng tuyên phạt Hồ Xuân Huy 12 năm tù."
    r = rouge(b, a)
    assert all(abs(v - 1) < 1e-9 for v in r.values()), f"Câu giống hệt phải = 1: {r}"
    r = rouge(unicodedata.normalize("NFD", b), a)
    assert all(abs(v - 1) < 1e-9 for v in r.values()), f"Lỗi chuẩn hoá NFC: {r}"
    r = rouge("mèo đen ngủ trên mái nhà", "công an bắt giữ đối tượng trộm xe")
    assert all(v == 0 for v in r.values()), f"Câu khác hẳn phải = 0: {r}"
    print("ROUGE self-test: OK")


# ======================== BASELINE LEAD-1 ========================
def lead1(article):
    """Câu đầu tiên của bài. VietNews tách sẵn ' . ' ở cuối câu,
    nên không bị nhầm với viết tắt như 'TP.'."""
    first = re.split(r"\s[.!?]\s", nfc(article).strip() + " ", maxsplit=1)[0]
    return detok(first + " .")


# ============ KIỂM TRA THÔNG TIN KHÔNG CÓ TRONG BÀI ============
NUM_RE = re.compile(r"\d+(?:[.,/:]\d+)*")


def numbers(text):
    return set(NUM_RE.findall(nfc(text)))


def names(text):
    """Cụm >= 2 âm tiết viết hoa liên tiếp (gần đúng tên riêng)."""
    t = re.sub(r"([,;:!?()\"“”'])", r" \1 ", nfc(text))
    out, cur = set(), []

    def flush():
        if len(cur) >= 2:
            out.add(tuple(words(" ".join(cur))))
        cur.clear()

    for tk in t.split():
        core = tk.rstrip(".")
        if core and core[0].isupper() and core.replace(".", "").isalpha():
            cur.append(core)
            if tk.endswith(".") and not core.isupper():   # hết câu (không phải 'TP.')
                flush()
        else:
            flush()
    flush()
    return out


def unsupported(pred, article):
    """Số và tên riêng có trong bản tóm tắt nhưng KHÔNG có trong bài báo."""
    art = detok(article)
    art_w = " " + " ".join(words(art)) + " "
    miss_num = sorted(x for x in numbers(pred) if x not in art)
    def found(seq):
        return " " + " ".join(seq) + " " in art_w

    # Bỏ qua chữ đầu cụm nếu chỉ là từ viết hoa đầu câu ("Ông Mai Văn Thương")
    miss_name = sorted(" ".join(n) for n in names(pred)
                       if not found(n) and not (len(n) >= 3 and found(n[1:])))
    return miss_num, miss_name


# ========================= SINH VĂN BẢN =========================
def load_model():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.padding_side = "left"      # bắt buộc khi sinh theo batch với mô hình decoder
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype="auto", device_map="auto"
    ).eval()
    return tok, model


def truncate(tok, text, n):
    ids = tok(text, add_special_tokens=False).input_ids
    return text if len(ids) <= n else tok.decode(ids[:n])


def build_prompt(tok, article):
    article = truncate(tok, detok(article), MAX_ARTICLE_TOKENS)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Nội dung bài báo:\n{article}"},
    ]
    return tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )


def generate_all(tok, model, articles, batch_size=BATCH_SIZE):
    import torch
    order = np.argsort([-len(a) for a in articles])  # bài dài trước: nếu OOM sẽ lộ ngay
    preds = [None] * len(articles)
    t0 = time.time()
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([build_prompt(tok, articles[i]) for i in idx],
                  return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, **GEN_KWARGS, pad_token_id=tok.pad_token_id)
        for j, i in enumerate(idx):
            ids = out[j, enc.input_ids.shape[1]:].tolist()
            if THINK_END_ID in ids:
                ids = ids[len(ids) - ids[::-1].index(THINK_END_ID):]
            preds[i] = tok.decode(ids, skip_special_tokens=True).strip()
        print(f"\r  {min(s + batch_size, len(order))}/{len(order)} mẫu | "
              f"{time.time() - t0:.0f}s", end="")
    print()
    return preds, time.time() - t0


# =========================== ĐÁNH GIÁ ===========================
def boot_ci(x, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    means = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return round(float(np.percentile(means, 2.5)), 2), round(float(np.percentile(means, 97.5)), 2)


def evaluate(name, df, preds, gen_seconds=None):
    rows = []
    for (_, ex), p in zip(df.iterrows(), preds):
        r = rouge(p, ex["abstract"])
        mn, mname = unsupported(p, ex["article"])
        rows.append({
            "guid": ex["guid"], "reference": ex["abstract"], "prediction": p,
            **{k: round(v * 100, 2) for k, v in r.items()},
            "n_words": len(words(p)),
            "unsupported_numbers": "|".join(mn), "unsupported_names": "|".join(mname),
            "has_unsupported": bool(mn or mname),
        })
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)

    summary = {"system": name, "n": len(res)}
    for k in ["rouge1", "rouge2", "rougeL"]:
        summary[k] = round(res[k].mean(), 2)
        summary[k + "_ci95"] = boot_ci(res[k])
    summary["avg_words"] = round(res["n_words"].mean(), 1)
    summary["pct_unsupported"] = round(res["has_unsupported"].mean() * 100, 1)
    if gen_seconds is not None:
        summary["sec_per_sample"] = round(gen_seconds / len(res), 3)
    return summary


def env_info(model):
    import datasets, torch, transformers
    info = {"torch": torch.__version__, "transformers": transformers.__version__,
            "datasets": datasets.__version__, "dtype": str(model.dtype),
            "batch_size": BATCH_SIZE, "gen_kwargs": GEN_KWARGS,
            "max_article_tokens": MAX_ARTICLE_TOKENS, "system_prompt": SYSTEM_PROMPT}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return info


if __name__ == "__main__":
    from datasets import load_dataset

    os.makedirs(OUT_DIR, exist_ok=True)
    self_test()

    test = load_dataset("Yuhthe/vietnews")["test"].shuffle(seed=SEED).select(range(N_TEST))
    df = test.to_pandas()   # tránh lỗi Column của datasets 4.x
    df[["guid"]].to_csv(os.path.join(OUT_DIR, "test_subset_guids.csv"), index=False)

    # 'reference' = chính abstract: ROUGE luôn 100, nhưng cho biết tỷ lệ
    # abstract chứa thông tin ngoài bài để so với mô hình.
    summaries = [evaluate("reference", df, list(df["abstract"]))]
    extra = {}

    if "lead1" in RUN_SYSTEMS:
        summaries.append(evaluate("lead1", df, [lead1(a) for a in df["article"]]))

    if "zeroshot" in RUN_SYSTEMS:
        tok, model = load_model()
        preds, secs = generate_all(tok, model, list(df["article"]))
        summaries.append(evaluate("zeroshot", df, preds, secs))
        extra = env_info(model)

    table = pd.DataFrame(summaries)
    print("\n" + table.to_string(index=False))
    with open(os.path.join(OUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
        json.dump({"results": summaries, "env": extra}, f, ensure_ascii=False, indent=2)
    print(f"\nĐã lưu kết quả vào thư mục {OUT_DIR}/")

Writing /kaggle/working/eval_baseline.py


In [4]:
!ls /kaggle/working
!pip install -q peft rouge-score
!cd /kaggle/working && python eval_baseline.py

eval_baseline.py  __notebook__.ipynb
ROUGE self-test: OK
model.safetensors.index.json: 100%|████████| 25.6k/25.6k [00:00<00:00, 27.3MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B /  622MB            
Reconstructing (incomplete total...):  17%|█▎      |  689MB / 4.06GB,  216MB/s  

Reconstructing (incomplete total...):  80%|██████▍ | 3.26GB / 4.06GB,  464MB/s  
Reconstructing (incomplete total...):  88%|███████ | 3.57GB / 4.06GB, 93.9MB/s  
Reconstructing (incomplete total...): 100%|████████| 4.06GB / 4.06GB,  191MB/s  

Fetching 2 files: 100%|███████████████████████████| 2/2 [00:21<00:00, 10.86s/it]
Download complete: : ███████████████████████████████████████| 3.53GB, 20.6MB/s  
Download complete: : ███████████████████████████████████████| 3.53GB, 20.6MB/s  
Reconstruction complete: 100%|███████████████████

In [5]:
import pandas as pd
df = pd.read_csv("results/zeroshot.csv")
cut = ~df["prediction"].str.strip().str[-1].isin(list(".!?\"”)"))
print(f"{cut.mean()*100:.1f}% không kết thúc bằng dấu câu")
print(df.loc[cut, "prediction"].head())

0.0% không kết thúc bằng dấu câu
Series([], Name: prediction, dtype: object)


In [6]:
%%writefile /kaggle/working/train_lora.py
"""Bước 6: fine-tune Qwen3-1.7B bằng LoRA trên VietNews (Kaggle, 1x T4).

Đặt file này CÙNG THƯ MỤC với eval_baseline.py (ví dụ /kaggle/working), rồi chạy trong notebook:

    !pip install -q peft rouge-score
    !python train_lora.py --mode overfit                     # 6a. kiểm tra pipeline (~10-15 phút)
    !python train_lora.py --mode train --variant raw         # E1: abstract gốc
    !python train_lora.py --mode train --variant filtered    # E2a: bỏ mẫu abstract có thông tin ngoài bài

Chạy bằng "!python" (không dán vào cell) để CUDA_VISIBLE_DEVICES có hiệu lực
và mỗi lần chạy có bộ nhớ GPU sạch.
"""
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle có 2 T4: chỉ dùng 1 cho đơn giản
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import argparse
import json
import math
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

import eval_baseline as EB   # dùng CHUNG prompt, detok, cấu hình sinh, ROUGE với baseline

SEED = 42
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# ============================ THAM SỐ ============================
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--mode", choices=["overfit", "train"], default="overfit")
    p.add_argument("--variant", choices=["raw", "filtered"], default="raw")
    p.add_argument("--train-size", type=int, default=5000)
    p.add_argument("--epochs", type=float, default=None)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--batch", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=8)
    p.add_argument("--dtype", choices=["fp16", "fp32"], default="fp16",
                   help="fp16: nạp trọng số fp16 (tiết kiệm bộ nhớ). "
                        "fp32: nạp fp32 + autocast fp16 (an toàn hơn nếu loss bị NaN).")
    p.add_argument("--out", default=None)
    a = p.parse_args()
    if a.mode == "overfit":
        a.train_size, a.grad_accum = 32, 1
        a.epochs = a.epochs or 30
        a.out = a.out or "runs/overfit"
    else:
        a.epochs = a.epochs or 1
        a.out = a.out or f"runs/{'E1_raw' if a.variant == 'raw' else 'E2a_filtered'}"
    return a


# ============================ DỮ LIỆU ============================
def select_train(n, variant):
    """Lấy n mẫu train cố định theo seed. 'filtered' bỏ mẫu có abstract chứa
    số/tên riêng không có trong bài, nhưng vẫn giữ ĐÚNG n mẫu để so sánh công bằng."""
    df = load_dataset("Yuhthe/vietnews")["train"].shuffle(seed=SEED).to_pandas()
    if variant == "raw":
        return df.iloc[:n].reset_index(drop=True), {"scanned": n, "dropped": 0}
    keep, scanned = [], 0
    for i, row in df.iterrows():
        scanned += 1
        miss_num, miss_name = EB.unsupported(row["abstract"], row["article"])
        if not (miss_num or miss_name):
            keep.append(i)
            if len(keep) == n:
                break
    return df.loc[keep].reset_index(drop=True), {"scanned": scanned, "dropped": scanned - n}


def build_example(tok, article, abstract):
    """input = prompt (y hệt lúc suy luận) + abstract + <|im_end|>.
    labels = -100 ở phần prompt -> chỉ tính loss trên bản tóm tắt."""
    prompt = EB.build_prompt(tok, article)            # đã detok + cắt article ở 1536 token
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    t_ids = tok(EB.detok(abstract) + "<|im_end|>", add_special_tokens=False).input_ids
    return {"input_ids": p_ids + t_ids, "labels": [-100] * len(p_ids) + t_ids}


def check_example(tok, ex, article):
    """Các kiểm tra bắt buộc trước khi train."""
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    n_prompt = sum(1 for l in ex["labels"] if l == -100)
    target = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]

    # 1. Prompt lúc train phải trùng KHỚP TỪNG TOKEN với prompt lúc sinh
    infer_ids = tok([EB.build_prompt(tok, article)]).input_ids[0]
    assert ex["input_ids"][:n_prompt] == infer_ids, "Prompt train khác prompt suy luận!"
    # 2. Phần tính loss phải kết thúc bằng <|im_end|> để mô hình học cách dừng
    assert target[-1] == im_end, "Thiếu <|im_end|> ở cuối nhãn!"
    # 3. Prompt phải kết thúc bằng khối <think> rỗng như lúc suy luận
    assert tok.decode(ex["input_ids"][:n_prompt]).endswith("<think>\n\n</think>\n\n")

    print(f"Tổng {len(ex['input_ids'])} token | prompt (bị che, -100): {n_prompt} "
          f"| được tính loss: {len(target)}")
    print("--- Cuối prompt (KHÔNG tính loss) ---")
    print(repr(tok.decode(ex["input_ids"][max(0, n_prompt - 25):n_prompt])))
    print("--- Phần TÍNH loss ---")
    print(repr(tok.decode(target)))
    print("Kiểm tra định dạng & masking: OK\n")


class Collator:
    """Pad bên phải. Token pad: input = pad_id, label = -100, attention = 0."""

    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        L = max(len(x["input_ids"]) for x in batch)
        ids = torch.full((len(batch), L), self.pad_id, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, x in enumerate(batch):
            n = len(x["input_ids"])
            ids[i, :n] = torch.tensor(x["input_ids"])
            lab[i, :n] = torch.tensor(x["labels"])
            att[i, :n] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}


# ============================= MÔ HÌNH =============================
def load_model(args):
    dtype = torch.float16 if args.dtype == "fp16" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        EB.MODEL_NAME, torch_dtype=dtype, device_map={"": 0}
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    cfg = LoraConfig(
        r=args.rank, lora_alpha=2 * args.rank,
        lora_dropout=0.0 if args.mode == "overfit" else 0.05,
        target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, cfg)
    # Tham số LoRA phải là fp32, nếu không fp16 AMP sẽ báo lỗi "unscale FP16 gradients"
    for p in model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    model.print_trainable_parameters()
    return model


@torch.no_grad()
def quick_generate(tok, model, df, title):
    """Sinh bằng ĐÚNG cấu hình của baseline, tính ROUGE và tỷ lệ thông tin ngoài bài."""
    model.eval()
    model.config.use_cache = True
    tok.padding_side = "left"
    torch.cuda.empty_cache()
    with torch.autocast("cuda", dtype=torch.float16):
        preds, secs = EB.generate_all(tok, model, list(df["article"]))
    r = [EB.rouge(p, ref) for p, ref in zip(preds, df["abstract"])]
    unsup = [any(EB.unsupported(p, a)) for p, a in zip(preds, df["article"])]
    res = {k: round(float(np.mean([x[k] for x in r])) * 100, 2) for k in ["rouge1", "rouge2", "rougeL"]}
    res.update(n=len(df), avg_words=round(float(np.mean([len(EB.words(p)) for p in preds])), 1),
               pct_unsupported=round(float(np.mean(unsup)) * 100, 1),
               sec_per_sample=round(secs / len(df), 3))
    print(f"\n===== {title} =====\n{res}")
    for i in range(3):
        print(f"\n[{i}] MODEL     : {preds[i]}\n    THAM CHIẾU: {EB.detok(df['abstract'][i])}")
    return res, preds


# ============================== MAIN ==============================
def main():
    args = parse_args()
    os.makedirs(args.out, exist_ok=True)
    torch.manual_seed(SEED)
    assert torch.cuda.is_available(), "Không thấy GPU: bật Accelerator -> GPU T4 trong Kaggle."
    print(f"GPU: {torch.cuda.get_device_name(0)} | mode={args.mode} | variant={args.variant}")

    tok = AutoTokenizer.from_pretrained(EB.MODEL_NAME)

    # ---------- dữ liệu ----------
    train_df, filt = select_train(args.train_size, args.variant)
    train_df[["guid"]].to_csv(os.path.join(args.out, "train_guids.csv"), index=False)
    if args.variant == "filtered":
        print(f"Lọc dữ liệu: duyệt {filt['scanned']} mẫu, bỏ {filt['dropped']} "
              f"({filt['dropped'] / filt['scanned'] * 100:.1f}%)")
    train = [build_example(tok, a, b) for a, b in zip(train_df["article"], train_df["abstract"])]
    check_example(tok, train[0], train_df["article"][0])

    eval_ds = None
    if args.mode == "train":
        val_df = load_dataset("Yuhthe/vietnews")["validation"].shuffle(seed=SEED).to_pandas()
        val_loss_df, val_gen_df = val_df.iloc[:200], val_df.iloc[200:300].reset_index(drop=True)
        eval_ds = [build_example(tok, a, b) for a, b in zip(val_loss_df["article"], val_loss_df["abstract"])]

    lens = [len(x["input_ids"]) for x in train]
    print(f"Độ dài chuỗi train: median={np.median(lens):.0f}, max={max(lens)}")

    # ---------- mô hình ----------
    model = load_model(args)

    steps = math.ceil(len(train) / (args.batch * args.grad_accum)) * args.epochs
    targs = TrainingArguments(
        output_dir=args.out,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch,
        gradient_accumulation_steps=args.grad_accum,
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        lr_scheduler_type="constant" if args.mode == "overfit" else "cosine",
        warmup_steps=0 if args.mode == "overfit" else max(1, int(0.05 * steps)),
        weight_decay=0.0,
        max_grad_norm=1.0,
        fp16=True,                          # T4 không hỗ trợ bf16
        logging_steps=10,
        eval_strategy="steps" if eval_ds else "no",
        eval_steps=50,
        save_strategy="steps" if args.mode == "train" else "no",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )
    trainer = Trainer(model=model, args=targs, train_dataset=train,
                      eval_dataset=eval_ds, data_collator=Collator(tok.pad_token_id))

    print(f"\nBắt đầu train: {len(train)} mẫu, {args.epochs} epoch, ~{int(steps)} bước tối ưu")
    t0 = time.time()
    trainer.train()
    train_secs = time.time() - t0
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    if any(not math.isfinite(l) for l in losses):
        print("\n!!! Loss bị NaN/inf. Chạy lại với --dtype fp32 (hoặc giảm --lr).")
        return

    model.save_pretrained(os.path.join(args.out, "adapter"))
    tok.save_pretrained(os.path.join(args.out, "adapter"))
    with open(os.path.join(args.out, "log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    # ---------- kiểm tra sau train ----------
    if args.mode == "overfit":
        res, _ = quick_generate(tok, model, train_df, "Sinh lại trên CHÍNH 32 mẫu đã học")
        ok = losses[-1] < 0.2 and res["rouge1"] >= 80
        print(f"\nLoss đầu: {losses[0]:.3f} -> cuối: {losses[-1]:.3f}")
        print("KẾT LUẬN:", "PASS - pipeline đúng, có thể train thật."
              if ok else "FAIL - xem lại masking/định dạng/learning rate trước khi train thật.")
    else:
        res, preds = quick_generate(tok, model, val_gen_df, "100 mẫu validation (xem nhanh)")
        pd.DataFrame({"guid": val_gen_df["guid"], "reference": val_gen_df["abstract"],
                      "prediction": preds}).to_csv(os.path.join(args.out, "val_preds.csv"), index=False)

    summary = {"args": vars(args), "filter": filt, "train_minutes": round(train_secs / 60, 1),
               "gpu_peak_gb": round(peak_gb, 2), "final_train_loss": losses[-1],
               "eval_losses": [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h],
               "quick_eval": res, "lora_targets": LORA_TARGETS,
               "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}
    with open(os.path.join(args.out, "train_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"\nThời gian train: {train_secs / 60:.1f} phút | GPU peak: {peak_gb:.2f} GB")
    print(f"Adapter và log đã lưu ở {args.out}/")


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_lora.py


In [7]:
# !pip uninstall -y torchao
# !cd /kaggle/working && python train_lora.py --mode overfit

In [8]:
!pip install -q peft rouge-score
!pip uninstall -y torchao
!pip list 2>/dev/null | grep -iE "^(torch|transformers|peft|datasets) "

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
datasets                              4.8.5
peft                                  0.20.0
torch                                 2.11.0+cu128
transformers                          5.16.1


In [9]:
!cd /kaggle/working && python train_lora.py --mode train --variant raw

GPU: Tesla T4 | mode=train | variant=raw
Tổng 1652 token | prompt (bị che, -100): 1594 | được tính loss: 58
--- Cuối prompt (KHÔNG tính loss) ---
' Đông) - Ảnh: TRẦN CÔNG ĐẠT<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- Phần TÍNH loss ---
'Hàng loạt những bất cập, hạn chế trong hình thức đầu tư BT đã được nêu ra tại hội thảo " Cơ chế đầu tư BT - Những vấn đề đặt ra và giải pháp hoàn thiện ", do Kiểm toán Nhà nước tổ chức hôm 19-10.<|im_end|>'
Kiểm tra định dạng & masking: OK

Độ dài chuỗi train: median=691, max=1770
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 311/311 [00:01<00:00, 162.59it/s]
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030

Bắt đầu train: 5000 mẫu, 1 epoch, ~313 bước tối ưu
{'loss': '2.76', 'grad_norm': '3.247', 'learning_rate': '0.00012', 'epoch': '0.032'}
{'loss': '1.584', 'grad_norm': '0.892', 'learning_rate': '0.0001999', 'epoch': '0.064'}
{'loss': 

In [10]:
!cd /kaggle/working && python train_lora.py --mode train --variant filtered

GPU: Tesla T4 | mode=train | variant=filtered
Lọc dữ liệu: duyệt 6506 mẫu, bỏ 1506 (23.1%)
Tổng 1652 token | prompt (bị che, -100): 1594 | được tính loss: 58
--- Cuối prompt (KHÔNG tính loss) ---
' Đông) - Ảnh: TRẦN CÔNG ĐẠT<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- Phần TÍNH loss ---
'Hàng loạt những bất cập, hạn chế trong hình thức đầu tư BT đã được nêu ra tại hội thảo " Cơ chế đầu tư BT - Những vấn đề đặt ra và giải pháp hoàn thiện ", do Kiểm toán Nhà nước tổ chức hôm 19-10.<|im_end|>'
Kiểm tra định dạng & masking: OK

Độ dài chuỗi train: median=692, max=1770
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 311/311 [00:01<00:00, 162.16it/s]
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030

Bắt đầu train: 5000 mẫu, 1 epoch, ~313 bước tối ưu
{'loss': '2.576', 'grad_norm': '3.168', 'learning_rate': '0.00012', 'epoch': '0.032'}
{'loss': '1.489', 'grad_norm': '1.042', 'learn

In [11]:
!cd /kaggle/working && zip -qr outputs.zip runs -x "*/checkpoint-*" && ls -lh outputs.zip

-rw-r--r-- 1 root root 128M Nov  4 21:17 outputs.zip
